# ADC Waveform Signal Extraction - ML Pipeline
Run the full ML pipeline on Google Colab with T4 GPU.

## 1. Setup: GPU check & clone repo

In [ ]:
# Cell 1: Verify GPU is available
!nvidia-smi

In [ ]:
# Cell 2: Clone the repository
!git clone https://github.com/ahoghmrt/ML_solution.git
%cd ML_solution

In [ ]:
# Cell 3: Install dependencies (most are pre-installed on Colab)
!pip install -q tensorflow scipy scikit-learn matplotlib pandas joblib

In [ ]:
# Cell 4: Verify TensorFlow sees the GPU
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")

## 2. Run the full pipeline

In [ ]:
# Cell 5: Run the full pipeline with GPU and PIT (permutation-invariant training)
!python cli.py run-all --experiment-name colab_t4_gpu --gpu --pit

## 3. View results

In [ ]:
# Cell 6: Show experiment metrics
import json, glob

exp_dirs = sorted(glob.glob("experiments/*colab_t4_gpu"))
if exp_dirs:
    with open(f"{exp_dirs[-1]}/metrics.json") as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))

In [ ]:
# Cell 7: Display training plots
from IPython.display import Image, display
import glob

for img in sorted(glob.glob("training_plots/*.png")):
    print(img)
    display(Image(filename=img))

In [ ]:
# Cell 8: Display comparison plots
for img in sorted(glob.glob("comparison_plots/*.png")):
    print(img)
    display(Image(filename=img))

In [ ]:
# Cell 9: Display a few example waveform predictions
for img in sorted(glob.glob("waveform_inspection/*.png"))[:6]:
    print(img)
    display(Image(filename=img))

In [ ]:
# Cell 10: Display error analysis plots
exp_dir = sorted(glob.glob("experiments/*colab_t4_gpu"))[-1]
for img in sorted(glob.glob(f"{exp_dir}/error_analysis/*.png")):
    print(img.split("/")[-1])
    display(Image(filename=img))

## 4. Save results to Google Drive

In [ ]:
# Cell 11: Mount Google Drive and copy all experiment results
from google.colab import drive
import shutil, glob

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/ML_solution_results"

# Copy experiment folder, models, plots, and scalers
exp_dir = sorted(glob.glob("experiments/*colab_t4_gpu"))[-1]
dirs_to_copy = {
    exp_dir: f"{save_dir}/{exp_dir.split('/')[-1]}",
    "training_plots": f"{save_dir}/training_plots",
    "comparison_plots": f"{save_dir}/comparison_plots",
    "waveform_inspection": f"{save_dir}/waveform_inspection",
}
files_to_copy = [
    "signal_count_model.keras",
    "signal_model.keras",
    "best_count_model.keras",
]

for src, dst in dirs_to_copy.items():
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Copied {src} -> {dst}")

for f in files_to_copy:
    shutil.copy2(f, f"{save_dir}/{f}")
    print(f"Copied {f}")

print(f"\nAll results saved to {save_dir}")

## 5. Download experiment files

Download all experiment results (models, plots, metrics, error analysis) as a zip file.

In [ ]:
# Cell 12: Download experiment files as zip
import shutil, glob, os
from google.colab import files

# Find the experiment directory
exp_dirs = sorted(glob.glob("experiments/*colab_t4_gpu"))
exp_dir = exp_dirs[-1] if exp_dirs else None

# Stage all results into a single folder for download
download_dir = "/content/ml_results"
os.makedirs(download_dir, exist_ok=True)

# Define what to collect
items = {
    "models/signal_count_model.keras": "signal_count_model.keras",
    "models/signal_model.keras": "signal_model.keras",
    "models/best_count_model.keras": "best_count_model.keras",
    "training_plots": "training_plots",
    "comparison_plots": "comparison_plots",
    "waveform_inspection": "waveform_inspection",
}
if exp_dir:
    items[f"experiment"] = exp_dir

# Copy everything into download staging dir
for dst_name, src in items.items():
    dst = os.path.join(download_dir, dst_name)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    elif os.path.isfile(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)

# Show what's included
print("Files staged for download:")
print("=" * 60)
for root, dirs, fnames in os.walk(download_dir):
    level = root.replace(download_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for f in sorted(fnames):
        fpath = os.path.join(root, f)
        size_kb = os.path.getsize(fpath) / 1024
        print(f"{subindent}{f}  ({size_kb:.1f} KB)")

# Create zip and trigger download
zip_path = shutil.make_archive("/content/ml_experiment_results", "zip", download_dir)
print(f"\nZip created: {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")
files.download(zip_path)